
# Yahoo A1 · Vanilla RNN (Supervised, Binary) · Point-level Anomaly Detection

- **Dataset path**: `./yahoo/A1Benchmark/real_*.csv`  
- **Model**: PyTorch `nn.RNN(input_size=1, hidden_size=H, num_layers=1, tanh)` → `Linear(H→1)`  
- **Task**: Supervised **binary classification** of anomalies at time `t` using past `L` values (`[t-L..t-1]`).  
- **Metrics**: Precision / Recall / F1 / Accuracy (point-level).  
- **Split per file**: 60% train / 20% val / 20% test (time order preserved).  
- **Scaling**: z-score using **train** μ,σ only.



## 0) Requirements
```
pip install torch numpy pandas scikit-learn
```


In [78]:
# 1) Configuration
from dataclasses import dataclass

@dataclass
class Config:
    a1_dir: str = "./TSB-AD-U/WSD"
    # a1_dir: str = "./yahoo/A1Benchmark"
    seq_len: int = 100
    hidden: int = 256
    lr: float = 1e-3
    batch: int = 256
    epochs: int = 1000
    patience: int = 15
    device: str = "cuda"  # "cuda" or "cpu"

cfg = Config()
cfg


Config(a1_dir='./TSB-AD-U/WSD', seq_len=100, hidden=256, lr=0.001, batch=256, epochs=1000, patience=15, device='cuda')

In [79]:

# 2) Imports
import random
import os, glob, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.5.1+cu124
CUDA available: True


## 3) Data utilities (Yahoo A1)

In [80]:

# Defensive column detection and CSV loader
def _find_cols(df: pd.DataFrame):
    ts_candidates = {"timestamp", "time", "ts", "date"}
    val_candidates = {"value", "values", "metric"}
    lab_candidates = {"is_anomaly", "anomaly", "label", "isoutlier"}
    ts_col = next((c for c in df.columns if str(c).lower() in ts_candidates), None)
    val_col = next((c for c in df.columns if str(c).lower() in val_candidates), None)
    lab_col = next((c for c in df.columns if str(c).lower() in lab_candidates), None)
    if val_col is None or lab_col is None:
        raise ValueError(f"Required columns not found in CSV. Columns: {df.columns.tolist()}")
    return ts_col, val_col, lab_col

def load_yahoo_a1(csv_path: str):
    df = pd.read_csv(csv_path)
    ts_col, val_col, lab_col = _find_cols(df)
    if ts_col is not None:
        df = df.sort_values(ts_col)
    x = df[val_col].to_numpy(dtype=np.float32)
    y = df[lab_col].astype(int).to_numpy()
    return x, y

def load_wsd(csv_path: str):
    df = pd.read_csv(csv_path)
    data_col = df.columns[0]
    label_col = df.columns[1]
    x = df[data_col].to_numpy(dtype=np.float32)
    y = df[label_col].astype(int).to_numpy()
    return x, y

def fit_robust_scaler(x_train: np.ndarray, cfg):
    # sklearn expects 2D; our series is 1D → reshape
    rs = RobustScaler()
    rs.fit(x_train.reshape(-1, 1))
    return rs

def apply_scaler(x: np.ndarray, scaler: RobustScaler):
    return scaler.transform(x.reshape(-1, 1)).astype(np.float32).squeeze(-1)

def make_windows(x: np.ndarray, y: np.ndarray, L: int):
    Xs, Ys, idx = [], [], []
    for t in range(L, len(x)):
        Xs.append(x[t - L:t])
        Ys.append(y[t])
        idx.append(t)
    X = np.asarray(Xs, dtype=np.float32)[..., None]  # [N, L, 1]
    Y = np.asarray(Ys, dtype=np.int64)               # [N]
    return X, Y, np.asarray(idx)

class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X)       # [N, L, 1]
        self.Y = torch.from_numpy(Y)       # [N]
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

def compute_pos_weight(y_train: np.ndarray) -> float:
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    if pos == 0:
        return 1.0
    return float(neg / pos)


## 4) Model (Vanilla RNN → Binary)

In [81]:

class VanillaRNNBinary(nn.Module):
    def __init__(self, hidden_size: int = 64):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            nonlinearity="tanh",
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)  # logit output

    def forward(self, x):           # x: [B, L, 1]
        out, _ = self.rnn(x)        # [B, L, H]
        h_last = out[:, -1, :]      # [B, H]
        logit = self.head(h_last)   # [B, 1]
        return logit.squeeze(-1)    # [B]

class VanillaLSTMBinary(nn.Module):
    def __init__(self, hidden_size: int = 64):
        super().__init__()
        # nn.RNN -> nn.LSTM
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,       # 1-layer (Vanilla)
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)  # 동일한 헤드

    def forward(self, x):          # x: [B, L, 1]
        # self.rnn(x) -> self.lstm(x)
        out, _ = self.lstm(x)     # out shape: [B, L, H]
        
        # 동일한 로직: 마지막 hidden state 사용
        h_last = out[:, -1, :]    # [B, H]
        
        logit = self.head(h_last)   # [B, 1]
        return logit.squeeze(-1)    # [B]
    
class VanillaGRUBinary(nn.Module):
    def __init__(self, hidden_size: int = 64):
        super().__init__()
        # nn.RNN -> nn.GRU
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,       # 1-layer (Vanilla)
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)  # 동일한 헤드

    def forward(self, x):          # x: [B, L, 1]
        # self.rnn(x) -> self.gru(x)
        out, _ = self.gru(x)      # out shape: [B, L, H]
        
        # 동일한 로직: 마지막 hidden state 사용
        h_last = out[:, -1, :]    # [B, H]
        
        logit = self.head(h_last)   # [B, 1]
        return logit.squeeze(-1)    # [B]

## 5) Train & Evaluation Functions

In [82]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for X, Y in loader:
        X = X.to(device)
        Y = Y.to(device).float()
        logit = model(X)
        loss = criterion(logit, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * X.size(0)
    return total / max(1, len(loader.dataset))

@torch.no_grad()
def infer_probs(model, loader, device):
    model.eval()
    probs, labels = [], []
    for X, Y in loader:
        X = X.to(device)
        p = torch.sigmoid(model(X)).cpu().numpy()
        probs.append(p)
        labels.append(Y.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def metrics_prfa(y_true: np.ndarray, y_prob: np.ndarray, thr: float = 0.5):
    y_pred = (y_prob >= thr).astype(np.int64)
    P = precision_score(y_true, y_pred, zero_division=0)
    R = recall_score(y_true, y_pred, zero_division=0)
    F1 = f1_score(y_true, y_pred, zero_division=0)
    # AUROC는 예측 확률값(y_prob)을 직접 사용
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:  # 단일 클래스만 있는 경우
        auroc = 0.0
    return {
        "Precision": float(P),
        "Recall": float(R),
        "F1": float(F1),
        "AUROC": float(auroc)
    }


## 6) Per-file Training & Testing Pipeline

In [83]:
# === 붙일 함수: F1이 최대가 되는 threshold 탐색 ===
import numpy as np
from sklearn.metrics import f1_score

def find_best_threshold(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)  # 0.05 ~ 0.95
    best_thr, best_f1 = 0.5, -1.0
    for thr in grid:
        f1 = f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr), float(best_f1)

def run_one_file(csv_path: str, cfg):
    # Load
    # x, y = load_yahoo_a1(csv_path)
    x, y = load_wsd(csv_path)
    n = len(x)
    if n < cfg.seq_len + 10:
        raise ValueError(f"Series too short for seq_len={cfg.seq_len}: {csv_path}")

    # Split (time order preserved): 60/20/20
    n_tr, n_va = int(0.6 * n), int(0.8 * n)
    x_tr, x_va, x_te = x[:n_tr], x[n_tr:n_va], x[n_va:]
    y_tr, y_va, y_te = y[:n_tr], y[n_tr:n_va], y[n_va:]

    # Scale using TRAIN only
    rs = fit_robust_scaler(x_tr, cfg)
    x_tr = apply_scaler(x_tr, rs)
    x_va = apply_scaler(x_va, rs)
    x_te = apply_scaler(x_te, rs)

    # Windows (labels at current t)
    L = cfg.seq_len
    Xtr, Ytr, _ = make_windows(x_tr, y_tr, L)
    Xva, Yva, _ = make_windows(np.concatenate([x_tr, x_va]),
                               np.concatenate([y_tr, y_va]), L)
    Xte, Yte, _ = make_windows(np.concatenate([x_tr, x_va, x_te]),
                               np.concatenate([y_tr, y_va, y_te]), L)

    # DataLoaders
    ds_tr, ds_va, ds_te = WindowDataset(Xtr, Ytr), WindowDataset(Xva, Yva), WindowDataset(Xte, Yte)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=cfg.batch, shuffle=False)
    dl_te = DataLoader(ds_te, batch_size=cfg.batch, shuffle=False)

    # Model / Loss / Optim
    device = cfg.device if (cfg.device == "cpu" or torch.cuda.is_available()) else "cpu"
    model = VanillaRNNBinary(hidden_size=cfg.hidden).to(device)
    # model = VanillaLSTMBinary(hidden_size=cfg.hidden).to(device)
    # model = VanillaGRUBinary(hidden_size=cfg.hidden).to(device)
    pos_w = compute_pos_weight(Ytr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-3)

    # Train with early stopping on val F1
    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, cfg.epochs + 1):
        _ = train_one_epoch(model, dl_tr, optimizer, criterion, device)
        p_va, y_va = infer_probs(model, dl_va, device)
        f1 = metrics_prfa(y_va, p_va)["F1"]
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        # Simple log
        if ep % 5 == 0 or bad == 0:
            print(f"[{os.path.basename(csv_path)}] epoch {ep} valF1={f1:.4f} best={best_f1:.4f}, bad={bad}")
        if bad >= cfg.patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    p_va, y_va = infer_probs(model, dl_va, device)
    best_thr, best_va_f1 = find_best_threshold(y_va, p_va)

    p_te, y_te = infer_probs(model, dl_te, device)
    return metrics_prfa(y_te, p_te, thr=best_thr)


## 7) Evaluate Over All A1 Files

In [84]:

def evaluate_all(cfg):
    # paths = sorted(glob.glob(os.path.join(cfg.a1_dir, "real_*.csv")))
    # if not paths:
    #     raise FileNotFoundError(f"No files matched: {cfg.a1_dir}/real_*.csv")
    
    paths = sorted(glob.glob(os.path.join(cfg.a1_dir, "*.csv")))
    if not paths:
        raise FileNotFoundError(f"No files matched: {cfg.a1_dir}/*.csv")
    rows = []
    for p in paths:
        print(f"\n### Processing: {p}")
        m = run_one_file(p, cfg)
        rows.append(m)
        print("=> Test metrics:", json.dumps(m, ensure_ascii=False))
    keys = rows[0].keys()
    macro = {k: float(np.mean([r[k] for r in rows])) for k in keys}
    return paths, rows, macro

paths, rows, macro = evaluate_all(cfg)

print("\n# Per-file metrics (Precision/Recall/F1/Accuracy)")
for p, m in zip(paths, rows):
    print(p, json.dumps(m, ensure_ascii=False))

print("\n# Macro mean (simple average over files)")
print(json.dumps(macro, ensure_ascii=False))



### Processing: ./TSB-AD-U/WSD/029_WSD_id_1_WebService_tr_4559_1st_10201.csv
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 1 valF1=0.0047 best=0.0047, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 2 valF1=0.0053 best=0.0053, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 3 valF1=0.0053 best=0.0053, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 5 valF1=0.0052 best=0.0053, bad=2
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 6 valF1=0.0060 best=0.0060, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 7 valF1=0.0063 best=0.0063, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 10 valF1=0.0062 best=0.0063, bad=3
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 13 valF1=0.0064 best=0.0064, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 14 valF1=0.0064 best=0.0064, bad=0
[029_WSD_id_1_WebService_tr_4559_1st_10201.csv] epoch 15 valF1=0.0063 best=0.0064, bad=1
[029_WSD_id_1_WebService_tr_4559_1st_1